# cancer-recon-apoptosis — RUNG 8: normal-tissue HLA-I heterogeneity (grounds RUNG-7's safety)

RUNG-7's gate-safety result rode on **one unsourced parameter** — the fraction of normal vital cells that are
*HLA-low* (so the Tmod blocker fails). **This run MEASURES it** from the CELLxGENE atlas, per vital cell type,
worst-donor, and feeds it back into RUNG-7 for a **data-grounded per-organ gate-toxicity floor.**

**Three things built in, by your ask:**
- **Resumable across the 4-hour cap** — one Drive tile per tissue; a disconnect loses nothing, just re-run Cell 5 tomorrow.
- **Foreground-visible logging** — a background `[heartbeat]` prints every ~20s so you always see the current step + RAM and know it isn't stuck.
- **GPU not used (by design)** — only 3 genes are pulled; the bottleneck is the Census fetch, so a GPU would sit idle. Run on a **CPU runtime** (saves your GPU quota).

Run cells top to bottom. After a disconnect: Cells 1, 2, 4, then **re-run Cell 5** — it resumes from Drive.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Google Drive for a RESUMABLE cache + start the run log
# Per-tissue HLA tiles are written to Drive. A 4-hour-cap disconnect loses NOTHING: re-run Cell 5 tomorrow
# and it SKIPS completed tissues and continues from where it stopped.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cancer-recon/rung8_tiles'
    os.makedirs(cache_dir, exist_ok=True)
    os.environ['RUNG8_CACHE'] = cache_dir
    print('[CELL 2] Drive mounted — RUNG8_CACHE =', cache_dir, '(tiles persist across days)')
except Exception as e:
    os.environ['RUNG8_CACHE'] = '/content/cancer-recon-apoptosis/runs/rung8_hla/tiles'
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — tiles in /content (LOST on disconnect!)')
from scripts.runlog import new_log
RUNLOG = new_log('rung8_hla', repo_dir='.')

In [ ]:
#@title Cell 3 — VALIDATE the math on synthetic ground truth (CPU, no data, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable, '-u', 'scripts/29_hla_heterogeneity.py', 'selftest'], RUNLOG)
print('[CELL 3]', '✓ validated — the aggregation + worst-donor + RUNG-7 coupling are correct'
      if rc == 0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install CELLxGENE Census (only needed for the real run)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓ (if Colab asks to RESTART runtime, do it, then Run all — the Drive cache makes resume instant)')

In [ ]:
#@title Cell 4b — GPU check (transparent: this rung does NOT need a GPU)
# RUNG-8 pulls only 3 genes (HLA-A/B/C). The bottleneck is the Census FETCH (network/disk); the per-donor
# aggregation is a trivial numpy groupby. So a GPU would sit idle — we don't bolt on unused GPU code.
# Recommendation: use a CPU runtime and save your free GPU quota for a rung that's compute-bound.
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
    print('[GPU]', out or '(none)', '— NOT used this rung (fetch-bound, 3 genes).')
except Exception as e:
    print('[GPU] none present (', type(e).__name__, ') — fine, RUNG-8 is CPU/fetch-bound.')

In [ ]:
#@title Cell 5 — REAL run: measure normal HLA-low fractions (RESUMABLE + heartbeat-logged)
#@markdown Light run (only 3 genes). A background **[heartbeat]** prints every ~20s so you ALWAYS see the
#@markdown current step + RAM and know it isn't stuck. Watch the **[N/9]** tissue progress. **If the 4-hour
#@markdown cap disconnects you, just RE-RUN THIS CELL** (after Cells 1,2,4) — completed tissues are on Drive
#@markdown and are skipped, so you continue exactly where you left off.
import sys, os
from scripts.runlog import run_logged
print('[CELL 5] starting — watch [+s][rung8] step lines and [heartbeat] every ~20s.')
print('         RESUMABLE: re-run this cell after any disconnect; Drive tiles ->', os.environ.get('RUNG8_CACHE'))
rc = run_logged([sys.executable, '-u', 'scripts/29_hla_heterogeneity.py', 'run'], RUNLOG)
print('[CELL 5]', '✓ done' if rc == 0 else f'✗ exit {rc} — just re-run this cell to resume from the last cached tissue')
from IPython.display import Image, display
import json
if os.path.exists('runs/rung8_hla/rung8_hla.png'):
    display(Image('runs/rung8_hla/rung8_hla.png'))
if os.path.exists('runs/rung8_hla/rung8_hla_heterogeneity.json'):
    d = json.load(open('runs/rung8_hla/rung8_hla_heterogeneity.json'))
    print('WORST vital type:', d['worst_vital_type'], '| HLA-A-low range:', d['worst_vital_hla_a_low_range'])
    print('-> data-grounded gate FPR range:', d['rung7_coupling'].get('data_grounded_FPR_at_lower'),
          'to', d['rung7_coupling'].get('data_grounded_FPR_at_upper'))
    print('   ranking (most->least HLA-low):', ', '.join(d.get('ranking_most_to_least_hla_low', [])[:6]))

In [ ]:
#@title Cell 6 — bundle the WHOLE run into ONE timestamped zip + download it
# One download per run (zip name carries <UTC-timestamp>_<git-sha>, matching the runlog + commit). Archive it
# on your Mac with:  python scripts/archive_colab_run.py --commit
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)                 # the log goes INTO the zip, not downloaded loose
stem = os.path.basename(str(RUNLOG)).replace('rung8_hla_', '').replace('.log', '')   # <UTC-ts>_<sha>
bundle = f'/content/rung8_run_{stem}.zip'
paths = sorted(set(glob.glob('runs/rung8_hla/*.png') + glob.glob('runs/rung8_hla/*.json') + [str(RUNLOG)]))
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if os.path.exists(p):
            z.write(p, arcname=os.path.basename(p)); print('  bundled', p)
print('[CELL 6] ->', bundle)
try:
    from google.colab import files; files.download(bundle)
except Exception as e:
    print('(download skipped:', type(e).__name__, e, ')')

## What RUNG-8 establishes
Measures the single parameter RUNG-7's safety result depended on — the **normal-tissue HLA-low fraction**,
per vital cell type, worst-donor — and feeds it back into RUNG-7 for a **data-grounded per-organ gate-toxicity
floor**. If the measured HLA-low fraction is **below** the assumed 5%, the blocker is more reliable than modelled
(safer gate); if **above**, RUNG-7 was optimistic and the gate is more toxic than thought.

**Honest ceiling:** mRNA HLA ≠ surface MHC-I protein; scRNA **dropout inflates** the HLA-low fraction (the
headline is an *upper bound* — conservatively over-estimating toxicity); HLA-I is **IFN-γ inducible** (resting
atlas may understate induced levels); scRNA resolves the **HLA-A gene**, not the **A\*02 allele**.